In [ ]:
# hide
import numpy as np
import pyquist as pq


def hann(n):
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / n))


def stft(x, hop, nF, window):
    return np.array([np.fft.rfft(x[s:s + nF] * window)
                     for s in range(0, len(x) - nF + 1, hop)])


def istft(S, hop, nF, window):
    out = np.zeros(hop * (S.shape[0] - 1) + nF)
    wsum = np.zeros_like(out)
    for k in range(S.shape[0]):
        out[k * hop:k * hop + nF] += np.fft.irfft(S[k], nF) * window
        wsum[k * hop:k * hop + nF] += window ** 2
    return out / np.maximum(wsum, 1e-8)

In [ ]:
# hide
from pyquist.helper import frequency_to_pitch, pitch_to_frequency

In [ ]:
# A tiny monophonic transcriber: for each frame, find the loudest frequency,
# round it to the nearest musical pitch, and emit a note when the pitch changes.
audio = pq.Audio.from_file("../assets/audio-melody.wav")
x = np.asarray(audio.samples).reshape(-1)
sr = audio.sample_rate
N_F, N_H = 4096, 1024

S = np.abs(stft(x, N_H, N_F, hann(N_F)))
freqs = np.fft.rfftfreq(N_F, 1 / sr)
seconds_per_frame = N_H / sr
threshold = 0.05 * S.max()

events, current, onset = [], None, 0.0
for k in range(S.shape[0]):
    frame = S[k]
    if frame.max() < threshold:              # silence
        pitch = None
    else:
        pitch = int(round(frequency_to_pitch(freqs[np.argmax(frame)])))
    if pitch != current:                     # the note changed
        if current is not None:
            events.append((onset, {"pitch": current, "duration": k * seconds_per_frame - onset}))
        current, onset = pitch, k * seconds_per_frame
if current is not None:                      # flush the final note
    events.append((onset, {"pitch": current, "duration": S.shape[0] * seconds_per_frame - onset}))

score = pq.Score(events)
for event in score:
    print(f"t = {event.time:4.2f}s   MIDI pitch {event.kwargs['pitch']}")